# Stage-B Ensemble v3 — Wider α Grid + Cross-Fitted Combiner

**Extends `two_stage_stageB_ensemble_v2.ipynb`** (current best: 7-member α-diverse pool + per-class
weight matrix, test macro-F1 **0.6302**).

## Three limitations of v2, all fixed by one change

| v2 limitation | Consequence | v3 fix |
|---|---|---|
| Base models trained on `train` only (195,950) | 20% of the available labelled data unused | Cross-fitting frees the whole of `train+val` |
| $W$ fitted on DSEL attack rows (**6,494**) | rarest families contribute ~20–30 rows; the same thinness that made thresholds overfit | $W$ fitted on **~32,000** OOF attack rows |
| α grid stopped at 1.0 | α=1.0 supplied most of the diversity — the axis was truncated exactly where it was paying | α extended to **1.5 and 2.0**, beyond full inverse frequency |

**Out-of-fold cross-fitting** delivers all three. Over `train+val` (244,938 rows, 64,936 attack), a
5-fold loop trains the pool on 4 folds and predicts the held-out fold, yielding honest
out-of-sample probabilities for *every* row. $W$ is then fitted on all of them, and the pool is
refit on the full data for deployment. Nothing is held back for the combiner, so no data is wasted.

This also matters because SEL was spent by the threshold study — v3 needs a fresh selection basis
and cross-fitting provides one from `train+val` without touching test.

## What α=1.5 and 2.0 mean

Weight ratio Worms:Exploits goes 128× (α=1) → ~1,400× (α=1.5) → ~16,000× (α=2). These are **expected
to be poor standalone models** — quite possibly degenerate, as focal γ=2 was. That is not the point.
v2's decisive lesson was that the member supplying most of the diversity (α=1.0) was individually
second-worst: **pool members are selected for complementarity, not quality.** §3 reports each
member's standalone score so any genuinely degenerate member is visible rather than hidden.

> **Runtime ≈ 10–12 min** (5 folds × 10 members + 1 gate, plus the final refit).

### Protocol

| Parameters | Fitted on |
|---|---|
| OOF probabilities | 5-fold cross-fit over `train+val` |
| $W$ (per-class weights) | **COMB** — half the OOF rows |
| variant choice | **SELV** — the other half, end-to-end via OOF gate |
| final pool + gate + $W$ | refit on all of `train+val` |
| evaluation | **test**, once |

## 1. Setup

In [1]:
%pip install xgboost imbalanced-learn -q

import numpy as np, pandas as pd, time, warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             classification_report, confusion_matrix)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

LABELS = {0:"Benign", 1:"Analysis", 2:"Backdoor", 3:"DoS", 4:"Exploits",
          5:"Fuzzers", 6:"Generic", 7:"Reconnaissance", 8:"Shellcode", 9:"Worms"}
NAMES = [LABELS[i] for i in range(10)]
RS = 42

X = pd.read_csv("Resource/CIC-UNSW-Data.csv")
y = pd.read_csv("Resource/CIC-UNSW-Label.csv")["Label"]
df = pd.concat([X, y], axis=1).drop_duplicates().reset_index(drop=True)
X, y = df[X.columns], df["Label"]

nun   = X.nunique(); zero = nun[nun <= 1].index.tolist()
corr  = X.drop(columns=zero).corr()
upper = corr.abs().where(np.triu(np.ones(corr.shape, bool), k=1))
red   = [c for c in upper.columns if (upper[c] >= 0.95).any()]
X = X.drop(columns=list(dict.fromkeys(zero + red)))

# same outer split as every other notebook -> identical test set
X_trf, X_test, y_trf, y_test = train_test_split(X, y, test_size=0.2, random_state=RS, stratify=y)
X_trf = X_trf.reset_index(drop=True); y_trf = y_trf.reset_index(drop=True)

attack_labels = sorted(l for l in y_trf.unique() if l != 0)
fam_to_idx = {f: i for i, f in enumerate(attack_labels)}
idx_to_fam = {i: f for f, i in fam_to_idx.items()}
NC  = len(attack_labels)
FAM = [LABELS[idx_to_fam[i]] for i in range(NC)]

y_arr   = y_trf.values
att_trf = y_arr != 0
fam_trf = np.full(len(y_arr), -1)
fam_trf[att_trf] = pd.Series(y_arr[att_trf]).map(fam_to_idx).values

print("features %d | train+val %d (%d attack) | test %d"
      % (X.shape[1], len(y_trf), att_trf.sum(), len(y_test)))
print("v2 used 195,950 base rows and 6,494 rows to fit W")

Note: you may need to restart the kernel to use updated packages.
features 44 | train+val 244938 (64936 attack) | test 61235
v2 used 195,950 base rows and 6,494 rows to fit W


In [2]:
def make_xgb(**kw):
    p = dict(n_estimators=400, max_depth=8, learning_rate=0.1, subsample=0.9,
             colsample_bytree=0.9, tree_method="hist", eval_metric="mlogloss",
             n_jobs=-1, random_state=RS)
    p.update(kw)
    return XGBClassifier(**p)

def sample_w(y_arr_, wc):
    w = wc[y_arr_].astype(float)
    return w * len(w) / w.sum()

def wc_alpha(n, a):
    w = (n.sum() / n) ** a
    return w / w.mean()

# --- the pool: alpha extended past 1.0, two SMOTE caps, plus RF/ET ---
SPECS = ([ (f"XGB a={a}", ("alpha", a)) for a in (0.0, 0.25, 0.5, 1.0, 1.5, 2.0) ]
         + [("XGB smote2000", ("smote", 2000)),
            ("XGB smote5000", ("smote", 5000)),
            ("RF a=0.25",     ("rf", 0.25)),
            ("ET a=0.25",     ("et", 0.25))])
MODELS = [s[0] for s in SPECS]; M = len(MODELS)
V2_SUBSET = ["XGB a=0.0", "XGB a=0.25", "XGB a=0.5", "XGB a=1.0",
             "XGB smote2000", "RF a=0.25", "ET a=0.25"]      # for the isolation control

def fit_member(kind, param, Xa, ya):
    """param is the alpha for alpha/rf/et, or the SMOTE cap for smote."""
    n = np.bincount(np.asarray(ya), minlength=NC)
    if kind == "smote":
        strat = {c: param for c in range(NC) if n[c] < param}
        Xs, ys = SMOTE(random_state=RS, sampling_strategy=strat).fit_resample(Xa, ya)
        m = make_xgb(); m.fit(Xs, ys); return m
    if   kind == "alpha": m = make_xgb()
    elif kind == "rf":    m = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=RS)
    elif kind == "et":    m = ExtraTreesClassifier(n_estimators=300, n_jobs=-1, random_state=RS)
    else: raise ValueError(kind)
    m.fit(Xa, ya, sample_weight=sample_w(np.asarray(ya), wc_alpha(n, param)))
    return m

def fit_gate(Xa, ya10):
    yb = (ya10 != 0).astype(int)
    g = make_xgb(); g.fit(Xa, yb, sample_weight=compute_sample_weight("balanced", yb))
    return g

print("pool size:", M, "members")

pool size: 10 members


## 2. Out-of-fold cross-fitting over `train+val`

Each fold trains the gate on all rows of the training part and the pool on its attack rows, then
predicts the held-out fold. Probabilities are stored for **every** row (not just attacks) because
end-to-end routing needs a family prediction wherever the gate fires.

Stored as `float32` — ten members × 244,938 × 9 would be ~176 MB in float64.

In [3]:
OOF      = [np.zeros((len(y_arr), NC), np.float32) for _ in range(M)]
oof_gate = np.zeros(len(y_arr), int)

t0 = time.time()
for f, (tr, va) in enumerate(StratifiedKFold(5, shuffle=True, random_state=RS).split(X_trf, y_arr)):
    tf = time.time()
    Xtr_f, ytr_f = X_trf.iloc[tr], y_arr[tr]
    oof_gate[va] = fit_gate(Xtr_f, ytr_f).predict(X_trf.iloc[va])

    m_att = ytr_f != 0
    Xa = Xtr_f[m_att].reset_index(drop=True)
    ya = pd.Series(fam_trf[tr][m_att])
    for mi, (nm, (kind, param)) in enumerate(SPECS):
        OOF[mi][va] = fit_member(kind, param, Xa, ya).predict_proba(X_trf.iloc[va]).astype(np.float32)
    print("  fold %d/5 done (%.0fs)" % (f + 1, time.time() - tf))
print("cross-fitting complete in %.0fs" % (time.time() - t0))

from sklearn.metrics import recall_score as _rs
print("OOF gate: attack recall %.4f / precision %.4f"
      % (_rs((y_arr != 0).astype(int), oof_gate),
         precision_score((y_arr != 0).astype(int), oof_gate)))

  fold 1/5 done (157s)
  fold 2/5 done (142s)
  fold 3/5 done (153s)
  fold 4/5 done (159s)
  fold 5/5 done (136s)
cross-fitting complete in 746s
OOF gate: attack recall 0.9961 / precision 0.9195


## 3. Did the wider α grid add diversity?

The v2 reference points to beat: mean pairwise disagreement **0.135**, oracle macro-F1 **0.7318**,
headroom **0.1456**. Standalone member scores are printed first so a degenerate α=2.0 member is
visible rather than silently poisoning the pool.

In [4]:
att_idx = np.where(att_trf)[0]
yf      = fam_trf[att_idx]
pred_a  = [OOF[m][att_idx].argmax(1) for m in range(M)]

print("--- standalone members (OOF, Stage-B on attack rows) ---")
print(pd.DataFrame([dict(model=nm,
        mF1=round(f1_score(yf, p, average="macro", zero_division=0), 4),
        mP =round(precision_score(yf, p, average="macro", zero_division=0), 4),
        mR =round(recall_score(yf, p, average="macro", zero_division=0), 4))
    for nm, p in zip(MODELS, pred_a)]).to_string(index=False))

D = np.zeros((M, M))
for i in range(M):
    for j in range(M):
        D[i, j] = (pred_a[i] != pred_a[j]).mean()
print("\nmean off-diagonal disagreement: %.4f   (v2: 0.1350)" % D[~np.eye(M, dtype=bool)].mean())
print(pd.DataFrame(D, index=MODELS, columns=MODELS).round(3).to_string())

any_right = np.zeros(len(yf), bool)
for p in pred_a:
    any_right |= (p == yf)
orc  = f1_score(yf, np.where(any_right, yf, pred_a[0]), average="macro", zero_division=0)
best = max(f1_score(yf, p, average="macro", zero_division=0) for p in pred_a)
print("\noracle %.4f | best single %.4f | headroom %.4f | any-correct %.2f%%"
      % (orc, best, orc - best, 100 * any_right.mean()))
print("v2 pool: oracle 0.7318 | best single 0.5863 | headroom 0.1456 | any-correct 83.52%")

--- standalone members (OOF, Stage-B on attack rows) ---
        model    mF1     mP     mR
    XGB a=0.0 0.5743 0.7004 0.5163
   XGB a=0.25 0.5859 0.6532 0.5483
    XGB a=0.5 0.5968 0.6239 0.5834
    XGB a=1.0 0.5630 0.5316 0.6210
    XGB a=1.5 0.4929 0.4569 0.6408
    XGB a=2.0 0.4090 0.4201 0.6353
XGB smote2000 0.5738 0.6648 0.5335
XGB smote5000 0.5613 0.6264 0.5352
    RF a=0.25 0.5732 0.6609 0.5241
    ET a=0.25 0.5565 0.6244 0.5142

mean off-diagonal disagreement: 0.2351   (v2: 0.1350)
               XGB a=0.0  XGB a=0.25  XGB a=0.5  XGB a=1.0  XGB a=1.5  XGB a=2.0  XGB smote2000  XGB smote5000  RF a=0.25  ET a=0.25
XGB a=0.0          0.000       0.061      0.089      0.204      0.367      0.530          0.053          0.063      0.124      0.153
XGB a=0.25         0.061       0.000      0.073      0.179      0.343      0.511          0.064          0.071      0.129      0.156
XGB a=0.5          0.089       0.073      0.000      0.152      0.315      0.487          0.090         

## 4. Fit the combiner on COMB, select on SELV

COMB and SELV are a stratified halving of `train+val`. $W$ is fitted on COMB's attack rows by the
same coordinate ascent as v2 — but on ~32,000 rows instead of 6,494, so a finer grid is affordable.

In [5]:
comb_idx, selv_idx = train_test_split(np.arange(len(y_arr)), test_size=0.5,
                                      random_state=RS + 1, stratify=y_arr)
comb_att = comb_idx[att_trf[comb_idx]]
print("COMB %d rows (%d attack) | SELV %d rows" % (len(comb_idx), len(comb_att), len(selv_idx)))
print("v2 fitted W on 6,494 attack rows -> now %dx more" % round(len(comb_att) / 6494))

GRID_W = (0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0)

def fuse(probas, W):
    S = np.zeros(probas[0].shape, np.float64)
    for m_ in range(len(probas)):
        S += probas[m_] * W[m_][None, :]
    return S

def fit_W(probas, ytrue, passes=3):
    mm = len(probas)
    W = np.ones((mm, NC)) / mm
    best = f1_score(ytrue, fuse(probas, W).argmax(1), average="macro", zero_division=0)
    for _ in range(passes):
        for c in range(NC):
            for m_ in range(mm):
                cur = W[m_, c]
                for g in GRID_W:
                    W[m_, c] = g
                    if W[:, c].sum() == 0:
                        continue
                    s = f1_score(ytrue, fuse(probas, W).argmax(1), average="macro", zero_division=0)
                    if s > best + 1e-7:
                        best, cur = s, g
                W[m_, c] = cur
    return W, best

sub = [MODELS.index(n) for n in V2_SUBSET]
t0 = time.time()
W_full,  s_full  = fit_W([OOF[m][comb_att] for m in range(M)], fam_trf[comb_att])
W_v2sub, s_v2sub = fit_W([OOF[m][comb_att] for m in sub],      fam_trf[comb_att])
print("W fitted in %.0fs | full pool COMB mF1 %.4f | v2-subset %.4f"
      % (time.time() - t0, s_full, s_v2sub))

COMB 122469 rows (32468 attack) | SELV 122469 rows
v2 fitted W on 6,494 attack rows -> now 5x more
W fitted in 42s | full pool COMB mF1 0.6195 | v2-subset 0.6197


In [6]:
def score10(fam_all, ye_arr, gate_pred, tag):
    """fam_all = Stage-B family index for every row; routed through the gate for the 10-class score."""
    yp = np.zeros(len(ye_arr), int)
    g = gate_pred == 1
    yp[g] = np.vectorize(idx_to_fam.get)(fam_all[g])
    return dict(variant=tag,
                e2e_mF1=round(f1_score(ye_arr, yp, average="macro", zero_division=0), 4),
                e2e_mP =round(precision_score(ye_arr, yp, average="macro", zero_division=0), 4),
                e2e_mR =round(recall_score(ye_arr, yp, average="macro", zero_division=0), 4),
                e2e_acc=round(float((yp == ye_arr).mean()), 4))

def evaluate_on(idx, probas_src, gate_src, ye_arr):
    P_full = [probas_src[m][idx] for m in range(M)]
    P_sub  = [probas_src[m][idx] for m in sub]
    gp     = gate_src[idx]
    out = [score10(fuse(P_full, np.ones((M, NC)) / M).argmax(1), ye_arr, gp, "static equal (10)"),
           score10(fuse(P_full, W_full).argmax(1),  ye_arr, gp, "per-class W (10 members)"),
           score10(fuse(P_sub,  W_v2sub).argmax(1), ye_arr, gp, "per-class W (v2 7-member subset)")]
    for m in range(M):
        out.append(score10(probas_src[m][idx].argmax(1), ye_arr, gp, "single: " + MODELS[m]))
    return pd.DataFrame(out)

selv_tab = evaluate_on(selv_idx, OOF, oof_gate, y_arr[selv_idx]).sort_values("e2e_mF1", ascending=False)
print(selv_tab.to_string(index=False))
BESTV = selv_tab.iloc[0]["variant"]
print("\nselected on SELV: %s" % BESTV)

                         variant  e2e_mF1  e2e_mP  e2e_mR  e2e_acc
per-class W (v2 7-member subset)   0.6312  0.6680  0.6452   0.9069
        per-class W (10 members)   0.6277  0.6647  0.6412   0.9079
               single: XGB a=0.5   0.6143  0.6224  0.6256   0.9059
               static equal (10)   0.6135  0.6122  0.6358   0.9059
              single: XGB a=0.25   0.6058  0.6470  0.5916   0.9076
               single: XGB a=0.0   0.5983  0.6888  0.5624   0.9077
               single: RF a=0.25   0.5976  0.6534  0.5722   0.9032
           single: XGB smote2000   0.5942  0.6573  0.5763   0.9078
           single: XGB smote5000   0.5868  0.6298  0.5811   0.9065
               single: ET a=0.25   0.5823  0.6183  0.5626   0.9007
               single: XGB a=1.0   0.5801  0.5402  0.6576   0.8949
               single: XGB a=1.5   0.5184  0.4775  0.6786   0.8736
               single: XGB a=2.0   0.4466  0.4505  0.6722   0.8413

selected on SELV: per-class W (v2 7-member subset)


### 4.1 Which member wins which family, and did α>1 earn its place?

If α=1.5 / 2.0 take meaningful weight on the rare families, extending the grid paid. If their columns
are ~0 everywhere, the α axis was already saturated at 1.0 and that is a clean finding too.

In [7]:
Wn = W_full / np.maximum(W_full.sum(0, keepdims=True), 1e-9)
print(pd.DataFrame(Wn, index=MODELS, columns=FAM).round(3).to_string())
print("\ndominant member per family:")
for c, f in enumerate(FAM):
    print("  %-15s %-14s (%.2f)" % (f, MODELS[Wn[:, c].argmax()], Wn[:, c].max()))
print("\ntotal weight mass per member (sum over families):")
print(pd.Series(Wn.sum(1), index=MODELS).round(3).sort_values(ascending=False).to_string())

               Analysis  Backdoor    DoS  Exploits  Fuzzers  Generic  Reconnaissance  Shellcode  Worms
XGB a=0.0         0.374     0.526  0.071     0.077    0.286    0.238           0.263      0.714  0.000
XGB a=0.25        0.280     0.000  0.071     0.077    0.057    0.000           0.105      0.048  0.111
XGB a=0.5         0.187     0.000  0.071     0.192    0.057    0.095           0.000      0.048  0.111
XGB a=1.0         0.019     0.000  0.071     0.077    0.143    0.095           0.000      0.048  0.111
XGB a=1.5         0.019     0.000  0.071     0.192    0.000    0.095           0.263      0.000  0.111
XGB a=2.0         0.019     0.000  0.071     0.077    0.286    0.095           0.263      0.000  0.111
XGB smote2000     0.019     0.105  0.357     0.077    0.000    0.095           0.000      0.000  0.111
XGB smote5000     0.019     0.000  0.071     0.077    0.057    0.095           0.000      0.048  0.111
RF a=0.25         0.019     0.263  0.071     0.077    0.057    0.095     

## 5. Refit on all of `train+val`, evaluate on test once

The pool and gate are refit on the full 244,938 rows; $W$ is refit on **all** OOF attack rows
(~64,936). Then a single pass over test.

> **Standard cross-fitting caveat:** $W$ is fitted against probabilities from models trained on 4/5
> of the data, but applied to models trained on 5/5, which are marginally sharper. This is inherent to
> stacking and is accepted practice; it biases slightly *against* v3, not for it.

In [ ]:
t0 = time.time()
W_final, s_final = fit_W([OOF[m][att_idx] for m in range(M)], yf)
print("W refit on all %d OOF attack rows -> mF1 %.4f (%.0fs)" % (len(att_idx), s_final, time.time() - t0))

t0 = time.time()
gate_f = fit_gate(X_trf, y_arr)
Xa_f = X_trf[att_trf].reset_index(drop=True)
ya_f = pd.Series(fam_trf[att_trf])
FINAL = [fit_member(k, p, Xa_f, ya_f) for _, (k, p) in SPECS]
print("pool refit on full train+val in %.0fs" % (time.time() - t0))

gate_test  = gate_f.predict(X_test)
P_test     = [m.predict_proba(X_test).astype(np.float32) for m in FINAL]
y_test_arr = y_test.values

# honour the SELV-selected variant -- do NOT hard-code one of them here
if BESTV == "per-class W (v2 7-member subset)":
    fam_pred = fuse([P_test[m] for m in sub], W_v2sub).argmax(1)
elif BESTV == "static equal (10)":
    fam_pred = fuse(P_test, np.ones((M, NC)) / M).argmax(1)
elif BESTV.startswith("single: "):
    fam_pred = P_test[MODELS.index(BESTV[len("single: "):])].argmax(1)
else:                                    # "per-class W (10 members)"
    fam_pred = fuse(P_test, W_final).argmax(1)

final = score10(fam_pred, y_test_arr, gate_test, "v3 selected: " + BESTV)
print("\n--- TEST (SELV-selected variant) ---")
print(pd.Series(final).to_string())
print("\nvs v2 ensemble (0.6302): %+.4f" % (final["e2e_mF1"] - 0.6302))
print("vs single a=0.25 train+val (0.6157): %+.4f" % (final["e2e_mF1"] - 0.6157))

In [9]:
test_tab = pd.DataFrame(
    [score10(fuse(P_test, np.ones((M, NC)) / M).argmax(1), y_test_arr, gate_test, "static equal (10)"),
     score10(fuse(P_test, W_final).argmax(1),  y_test_arr, gate_test, "per-class W (10 members)"),
     score10(fuse([P_test[m] for m in sub], W_v2sub).argmax(1), y_test_arr, gate_test,
             "per-class W (v2 7-member subset)")]
    + [score10(P_test[m].argmax(1), y_test_arr, gate_test, "single: " + MODELS[m]) for m in range(M)]
).sort_values("e2e_mF1", ascending=False)
print(test_tab.to_string(index=False))

print("\nSELV order : %s" % " > ".join(list(selv_tab.variant)[:4]))
print("test  order: %s" % " > ".join(list(test_tab.variant)[:4]))
print("SELV-selected '%s' ranks %d of %d on test"
      % (BESTV, 1 + list(test_tab.variant).index(BESTV), len(test_tab)))

                         variant  e2e_mF1  e2e_mP  e2e_mR  e2e_acc
per-class W (v2 7-member subset)   0.6339  0.6701  0.6475   0.9073
        per-class W (10 members)   0.6329  0.6796  0.6425   0.9065
              single: XGB a=0.25   0.6157  0.6574  0.6035   0.9081
               static equal (10)   0.6113  0.6087  0.6357   0.9058
               single: XGB a=0.5   0.6074  0.6076  0.6264   0.9063
               single: XGB a=0.0   0.6028  0.6887  0.5702   0.9084
           single: XGB smote2000   0.5976  0.6624  0.5790   0.9080
           single: XGB smote5000   0.5927  0.6390  0.5848   0.9071
               single: XGB a=1.0   0.5858  0.5427  0.6779   0.8944
               single: RF a=0.25   0.5853  0.6393  0.5610   0.9025
               single: ET a=0.25   0.5652  0.6055  0.5467   0.8997
               single: XGB a=1.5   0.5064  0.4688  0.6835   0.8716
               single: XGB a=2.0   0.4474  0.4530  0.6845   0.8416

SELV order : per-class W (v2 7-member subset) > per-class W (

In [10]:
yp = np.zeros(len(y_test_arr), int)
g = gate_test == 1
yp[g] = np.vectorize(idx_to_fam.get)(fam_pred[g])
print(classification_report(y_test_arr, yp, labels=list(range(10)),
                            target_names=NAMES, digits=3, zero_division=0))
print(pd.DataFrame(confusion_matrix(y_test_arr, yp, labels=list(range(10))),
                   index=NAMES, columns=NAMES).to_string())

v2_f1 = dict(zip(NAMES, [0.983, 0.427, 0.652, 0.433, 0.750, 0.708, 0.790, 0.641, 0.360, 0.557]))
v3_f1 = f1_score(y_test_arr, yp, average=None, labels=range(10), zero_division=0)
print("\nper-class F1 vs v2:")
print(pd.DataFrame({"v2": [v2_f1[n] for n in NAMES], "v3": v3_f1.round(3),
                    "delta": (v3_f1 - np.array([v2_f1[n] for n in NAMES])).round(3)},
                   index=NAMES).to_string())

                precision    recall  f1-score   support

        Benign      0.999     0.968     0.983     45001
      Analysis      0.315     0.818     0.455        77
      Backdoor      0.836     0.511     0.634        90
           DoS      0.538     0.404     0.462       883
      Exploits      0.868     0.696     0.773      6182
       Fuzzers      0.567     0.942     0.708      5279
       Generic      0.838     0.774     0.805       899
Reconnaissance      0.778     0.568     0.657      2406
     Shellcode      0.366     0.368     0.367       370
         Worms      0.692     0.375     0.486        48

      accuracy                          0.906     61235
     macro avg      0.680     0.642     0.633     61235
  weighted avg      0.925     0.906     0.910     61235

                Benign  Analysis  Backdoor  DoS  Exploits  Fuzzers  Generic  Reconnaissance  Shellcode  Worms
Benign           43546        51         2   18        73     1207       13              59         32 

## 6. Findings

**v3 = test macro-F1 0.6339**, selected on SELV, ranked **1st of 13 on test**. New project best, but
the gain over v2 is small: **+0.0037**.

| Configuration | SELV | **Test** |
|---|---|---|
| **per-class W, v2 7-member subset** *(SELV-selected)* | **0.6312** | **0.6339** |
| per-class W, 10 members | 0.6277 | 0.6329 |
| single XGB α=0.25 | 0.6058 | 0.6157 |
| static equal (10) | 0.6135 | 0.6113 |
| v2 ensemble (reference) | — | 0.6302 |

> **Reporting note:** the `--- TEST ---` block in §5 evaluates the **10-member** variant (0.6329)
> because that cell was hard-coded to `W_final` rather than to the SELV-selected variant. The
> selected variant is the 7-member subset at **0.6339**, visible in §5.1's full table. Both are
> correct evaluations; only the highlighted one was wrong. The cell has been fixed for re-runs.

The SELV→test ordering held (top two identical, selected variant 1st of 13), so unlike the threshold
study this gain is not selection noise.

### 1. Extending α past 1.0 produced enormous diversity — and none of it converted

The mechanism worked exactly as intended on every diversity measure:

| Pool | mean pairwise disagreement | oracle macro-F1 | headroom |
|---|---|---|---|
| v2 (7 members, α ≤ 1) | 0.135 | 0.7318 | 0.1456 |
| **v3 (10 members, α ≤ 2)** | **0.235** | **0.8218** | **0.2250** |

Disagreement rose 74%, oracle headroom by 55%. α=2.0 disagrees with α=0.0 on **53% of attack rows** —
these are genuinely different classifiers.

**And the 10-member pool scored *below* the 7-member subset** (0.6329 vs 0.6339). The two extra α
members plus SMOTE-5000 contributed nothing net, despite the combiner assigning them real weight
(α=2.0 received the second-highest total mass, 0.922, and took 0.26–0.29 on Reconnaissance and
Fuzzers).

**This is the substantive result of the notebook: oracle headroom is a poor predictor of realisable
gain.** Between v2 and v3 the theoretical ceiling rose 9 points and the achieved score moved
+0.004. It also retrospectively explains DES v1 — that pool had 0.0804 of headroom and realised
none; the assumption that headroom implies opportunity was wrong then and is wrong here. **The α
axis saturates at ≈1.0**: past that point the members disagree more but are not *usefully* different,
because α=1.5/2.0 are simply degraded (standalone macro-F1 0.4929 and 0.4090) rather than
complementary.

### 2. So where did the +0.0037 actually come from?

The `v2 7-member subset` row is the control that answers this: **identical members to v2**, differing
only in (a) base models cross-fitted over `train+val` rather than trained on `train`, and (b) $W$
fitted on 32,468 OOF rows rather than 6,494. It scored 0.6339 vs v2's 0.6302.

So the entire gain is from **more data**, not from the wider α grid. Both data improvements are worth
keeping — they are free at inference — but they are worth ~0.4 points, not a step change.

### 3. Per-class movement (10-member variant vs v2)

| Family | v2 | v3 | Δ |
|---|---|---|---|
| DoS | 0.433 | 0.462 | **+0.029** |
| Analysis | 0.427 | 0.455 | +0.028 |
| Exploits | 0.750 | 0.773 | +0.023 |
| Reconnaissance | 0.641 | 0.657 | +0.016 |
| Generic | 0.790 | 0.805 | +0.015 |
| Shellcode | 0.360 | 0.367 | +0.007 |
| Fuzzers | 0.708 | 0.708 | 0.000 |
| Backdoor | 0.652 | 0.634 | −0.018 |
| **Worms** | 0.557 | 0.486 | **−0.071** |

Broad small gains across seven families, paid for almost entirely by **Worms** — the class with 48
test rows, where three or four flows move F1 by ~0.07. That is within the noise band for a class that
small, and it is the honest caveat on the headline: with Worms at 48 rows and Analysis at 77, the
macro average is genuinely unstable at the third decimal. **Differences of ±0.005 between
configurations should not be over-interpreted**, which also means v3-over-v2 is suggestive rather than
decisively established.

Analysis is the interesting one: recall 0.818 (63 of 77 caught) at precision 0.315. The ensemble is
now willing to over-call Analysis to catch it — a defensible NIDS trade, but one to state explicitly.

### 4. Cross-fitting cost

746 s for the 5-fold loop plus 180 s for the final pool refit — ~16 minutes against v2's ~1 minute,
for +0.0037. Worth it once for a final thesis number; not worth it as an iteration loop.

### Recommendation

**Adopt v3's 7-member configuration as the final Stage-B model (0.6339)** — it is the SELV-selected
variant, it ranked 1st on test, and it costs the same at inference as v2 (seven models, one 7×9
matrix). The cross-fitting expense is training-time only.

**Do not extend the α grid further.** §1 bounds the mechanism: the axis is saturated at 1.0 and more
extreme weighting produces disagreement without complementarity.

### Where this leaves the project

Improvement has clearly plateaued: 0.6157 → 0.6302 → 0.6339, with the last step inside the noise band
of the rarest classes. Four separate levers (focal weighting, dynamic ensembling, decision
thresholds, wider α) have now returned ~nothing, while every gain has come from either better
weighting or more data.

That pattern is itself the finding, and it points at the same place three previous notebooks did:
**the residual error is shared across models, weightings, pools and decision rules.** The
label-ambiguity ceiling analysis would measure directly whether that residual is irreducible — and at
this point it is better motivated than any further modelling change. **SHAP** also remains outstanding
in the novelty stack.
